# Part B – Question 3: Linear and Kernel Regression on Diabetes Dataset

We use `scikit-learn` (permitted for this question) to compare:
1. Ordinary Least Squares (OLS) linear regression
2. Kernel Ridge Regression with two different kernels (RBF and Polynomial)

Both models are evaluated on the **diabetes** regression dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## 1. Load and Prepare Data

In [ ]:
data = load_diabetes()
X, y = data.data, data.target

# 80/20 split, fixed seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardise features (important for kernel methods)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape},  Test: {X_test.shape}")

## 2. (a) Ordinary Least Squares Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_mse  = np.mean((y_test - lr_pred) ** 2)

print(f"Linear Regression  MSE = {lr_mse:.2f}")

## 3. (b) Kernel Ridge Regression

We try two kernels:

* **RBF (Gaussian)**: $K(x, y) = \exp\!\left(-\gamma\|x-y\|^2\right)$
* **Polynomial**: $K(x, y) = (\gamma\, x^T y + c_0)^d$

The regularisation strength `alpha` is kept the same for both.

In [ ]:
# ---- RBF Kernel ----
krr_rbf = KernelRidge(kernel='rbf', alpha=1.0, gamma=0.1)
krr_rbf.fit(X_train_s, y_train)
rbf_pred = krr_rbf.predict(X_test_s)
rbf_mse  = np.mean((y_test - rbf_pred) ** 2)
print(f"Kernel Ridge (RBF,         gamma=0.1)  MSE = {rbf_mse:.2f}")

# ---- Polynomial Kernel ----
krr_poly = KernelRidge(kernel='polynomial', alpha=1.0, degree=3, gamma=0.1, coef0=1)
krr_poly.fit(X_train_s, y_train)
poly_pred = krr_poly.predict(X_test_s)
poly_mse  = np.mean((y_test - poly_pred) ** 2)
print(f"Kernel Ridge (Polynomial,  degree=3)   MSE = {poly_mse:.2f}")

## 4. Summary Table

In [ ]:
print(f"{'Model':<35}{'Test MSE':>12}")
print("-" * 48)
print(f"{'Linear Regression (OLS)':<35}{lr_mse:>12.2f}")
print(f"{'Kernel Ridge (RBF, gamma=0.1)':<35}{rbf_mse:>12.2f}")
print(f"{'Kernel Ridge (Poly, d=3)':<35}{poly_mse:>12.2f}")

## 5. Prediction Plots

In [ ]:
models = [
    ('Linear Regression', lr_pred, lr_mse),
    ('Kernel Ridge (RBF)', rbf_pred, rbf_mse),
    ('Kernel Ridge (Poly)', poly_pred, poly_mse),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, preds, mse_val) in zip(axes, models):
    ax.scatter(y_test, preds, alpha=0.6, s=25)
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'r--', label='Perfect fit')
    ax.set_xlabel('True value')
    ax.set_ylabel('Predicted value')
    ax.set_title(f'{name}\nMSE = {mse_val:.2f}')
    ax.legend()

plt.tight_layout()
plt.savefig('q3_regression_comparison.png', dpi=100)
plt.show()